# Roslynator Maintainability — Nesting Depth Raw Output Extraction (C#)

This notebook analyzes **C# repositories** with **Roslynator** and a custom **Roslyn AST nesting-depth analyzer**, capturing complete raw tool output for maintainability metric derivation and validation.

**Default benchmark repository:** [dotnet/roslyn](https://github.com/dotnet/roslyn)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.

**Prerequisites:** .NET SDK 8+ and Roslynator CLI (installed automatically when missing).


## Section 1 — Install Dependencies

Install open-source Python packages, bootstrap the .NET SDK, and install Roslynator CLI.


In [ ]:
!pip install -q pandas gitpython jupyter


In [ ]:
import os
import sys
from pathlib import Path

os.environ.pop('PYTHONPATH', None)
METRIC_ROOT = Path('.').resolve()
TOOL_ROOT = METRIC_ROOT / 'tool'
PROJECT_ROOT = METRIC_ROOT.parent.parent
RUNTIMES_ROOT = PROJECT_ROOT / 'runtimes'
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from run_roslynator_benchmark_impl import (
    DOTNET_CHANNEL,
    build_nesting_analyzer,
    download_dotnet_sdk,
    dotnet_env,
    dotnet_executable,
    install_roslynator,
    run_command,
)

DOTNET_ROOT = (RUNTIMES_ROOT / 'dotnet-sdk').resolve()
DOTNET_TOOLS_DIR = (RUNTIMES_ROOT / 'dotnet-tools').resolve()

DOTNET_ROOT = download_dotnet_sdk(DOTNET_ROOT, channel=DOTNET_CHANNEL)
ROSLYNATOR_PATH = install_roslynator(DOTNET_ROOT, DOTNET_TOOLS_DIR)
ANALYZER_DLL = build_nesting_analyzer(DOTNET_ROOT)

version_stdout, version_stderr, _ = run_command(
    [str(ROSLYNATOR_PATH), '--version'],
    env=dotnet_env(DOTNET_ROOT),
)
print((version_stdout or version_stderr).strip())
print(f'.NET SDK: {dotnet_executable(DOTNET_ROOT)}')
print(f'Roslynator: {ROSLYNATOR_PATH}')
print(f'NestingDepthAnalyzer: {ANALYZER_DLL}')


## Section 2 — Configuration

Set execution mode, repository source, and output location.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/dotnet/roslyn.git'

LOCAL_REPO_PATH = '/content/roslyn'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

WORKSPACE_DIR = './workspace'

STREAM_RAW_OUTPUT = True

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark (100% predictable nesting outcomes):
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/cs_nesting_benchmark'


## Utility Functions

Modular helpers for logging, repository setup, Roslynator execution, and nesting-depth extraction.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.pop("PYTHONPATH", None)
METRIC_ROOT = Path.cwd().resolve()
TOOL_ROOT = METRIC_ROOT / "tool"
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

from run_roslynator_benchmark_impl import (
    analyze_targets,
    build_nesting_analyzer,
    compute_summary,
    discover_csharp_files,
    discover_solution_and_project_files,
    dotnet_env,
    install_roslynator,
    merge_roslynator_results,
    parse_roslynator_text,
    parse_roslynator_xml,
    run_nesting_analyzer,
    run_roslynator_suite,
)

CS_EXTENSIONS = {".cs"}
EXCLUDED_DIR_NAMES = {
    ".git",
    "bin",
    "obj",
    "packages",
    "artifacts",
    "TestResults",
    "node_modules",
}


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')

    logger.info(f"Cloning {repo_url} into {clone_path} (depth={clone_depth})")
    try:
        clone_kwargs = {"depth": clone_depth} if clone_depth else {}
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Clone failed for {repo_url}: {exc}")
        raise

    logger.info(f"Clone completed: {clone_path}")
    return clone_path.resolve()


def validate_local_repository(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        message = f"Local repository path does not exist: {local_repo_path}"
        logger.error(message)
        raise FileNotFoundError(message)

    if not local_repo_path.is_dir():
        message = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(message)
        raise NotADirectoryError(message)

    has_git = (local_repo_path / ".git").exists()
    has_cs = any(
        file_path.suffix.lower() in CS_EXTENSIONS
        for file_path in local_repo_path.rglob("*")
        if file_path.is_file()
    )

    if not has_git and not has_cs:
        message = (
            f"Path is neither a Git repository nor a C# source directory: {local_repo_path}"
        )
        logger.error(message)
        raise ValueError(message)

    if has_git:
        try:
            Repo(local_repo_path)
        except InvalidGitRepositoryError as exc:
            logger.error(f"Invalid Git repository at {local_repo_path}: {exc}")
            raise

    logger.info(f"Validated local repository path: {local_repo_path}")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        logger.info("Execution mode: Git repository URL")
        return clone_or_reuse_repository(
            repo_url, workspace_dir, if_clone_exists, logger, clone_depth=clone_depth
        )
    logger.info("Execution mode: Local repository path")
    return validate_local_repository(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def compute_repository_stats(repo_path: Path, cs_files: list[Path]) -> dict[str, Any]:
    total_size_bytes = sum(file_path.stat().st_size for file_path in cs_files)
    directory_count = sum(
        1
        for current_path, _, _ in os.walk(repo_path)
        if not should_exclude_path(Path(current_path).relative_to(repo_path))
    )
    return {
        "csharp_file_count": len(cs_files),
        "repository_size_bytes": total_size_bytes,
        "directory_count": directory_count,
    }


def save_csharp_file_list(cs_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    rows = [
        {
            "absolute_path": str(file_path),
            "relative_path": str(file_path.relative_to(repo_path)),
        }
        for file_path in cs_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW ROSLYNATOR OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


def count_failed_methods(nesting_df: pd.DataFrame) -> int:
    if nesting_df.empty or "status" not in nesting_df.columns:
        return 0
    return int((nesting_df["status"] != "analyzed").sum())



## Section 3 — Repository Setup

Resolve the repository path based on configuration and initialize output directories.


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

logger.info(f'Repository ready at: {REPO_PATH}')


## Section 4 — Discover C# Files

Recursively discover `.cs` files while excluding build and dependency directories.


In [ ]:
CS_FILES = discover_csharp_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, CS_FILES)

CSHARP_FILES_CSV = OUTPUT_PATH / 'csharp_files.csv'
save_csharp_file_list(CS_FILES, REPO_PATH, CSHARP_FILES_CSV)

print(f'Total C# Files Found: {len(CS_FILES)}')
print(f'Repository Size (C# files only): {REPO_STATS["repository_size_bytes"]:,} bytes')
print(f'Total Directories (excluding filtered paths): {REPO_STATS["directory_count"]:,}')
print(f'Saved file list to: {CSHARP_FILES_CSV}')


## Section 5 — Execute Roslynator

Discover `.sln` and `.csproj` files and analyze each solution/project sequentially.

Example equivalent command:

```bash
roslynator analyze <solution_or_project_path>
```

Raw stdout and stderr are preserved without suppression.


In [ ]:
SOLUTIONS, PROJECTS = discover_solution_and_project_files(REPO_PATH)
ANALYSIS_TARGETS = analyze_targets(SOLUTIONS, PROJECTS)
ENV = dotnet_env(DOTNET_ROOT)

if not CS_FILES:
    logger.error('No C# files discovered; skipping Roslynator execution.')
    ROSLYNATOR_RAW_TEXT = ''
    ROSLYNATOR_XML_PATHS = []
elif not ANALYSIS_TARGETS:
    logger.error('No .sln or .csproj files discovered; Roslynator analysis skipped.')
    ROSLYNATOR_RAW_TEXT = ''
    ROSLYNATOR_XML_PATHS = []
else:
    logger.info(f'Roslynator targets: {len(ANALYSIS_TARGETS)}')
    for target in ANALYSIS_TARGETS:
        logger.info(f'  - {target}')
    if STREAM_RAW_OUTPUT:
        print('\nStreaming Roslynator raw output...\n')
    ROSLYNATOR_RAW_TEXT, ROSLYNATOR_XML_PATHS = run_roslynator_suite(
        roslynator=ROSLYNATOR_PATH,
        targets=ANALYSIS_TARGETS,
        xml_output=OUTPUT_PATH / 'roslynator_output.xml',
        env=ENV,
    )
    if STREAM_RAW_OUTPUT:
        print(ROSLYNATOR_RAW_TEXT)

logger.info(f'Roslynator execution complete. Raw output size: {len(ROSLYNATOR_RAW_TEXT):,} characters')


## Section 6 — Raw Output Extraction

Persist complete raw Roslynator text output, XML output, and a CSV representation of all diagnostics.


In [ ]:
import xml.etree.ElementTree as ET

RAW_OUTPUT_PATH = OUTPUT_PATH / 'roslynator_raw_output.txt'
XML_OUTPUT_PATH = OUTPUT_PATH / 'roslynator_output.xml'
RESULTS_CSV_PATH = OUTPUT_PATH / 'roslynator_results.csv'

RAW_OUTPUT_PATH.write_text(ROSLYNATOR_RAW_TEXT, encoding='utf-8')

if ROSLYNATOR_XML_PATHS:
    if len(ROSLYNATOR_XML_PATHS) == 1 and ROSLYNATOR_XML_PATHS[0] != XML_OUTPUT_PATH:
        XML_OUTPUT_PATH.write_text(ROSLYNATOR_XML_PATHS[0].read_text(encoding='utf-8'), encoding='utf-8')
    elif len(ROSLYNATOR_XML_PATHS) > 1:
        combined_root = ET.Element('Roslynator')
        code_analysis = ET.SubElement(combined_root, 'CodeAnalysis')
        projects = ET.SubElement(code_analysis, 'Projects')
        for xml_path in ROSLYNATOR_XML_PATHS:
            parsed_root = ET.parse(xml_path).getroot()
            for project_node in parsed_root.iter():
                if str(project_node.tag).endswith('Project'):
                    projects.append(project_node)
        ET.ElementTree(combined_root).write(XML_OUTPUT_PATH, encoding='utf-8', xml_declaration=True)
elif not XML_OUTPUT_PATH.exists():
    XML_OUTPUT_PATH.write_text('<?xml version="1.0" encoding="utf-8"?><Roslynator/>', encoding='utf-8')

xml_df = parse_roslynator_xml([XML_OUTPUT_PATH] if XML_OUTPUT_PATH.exists() else [])
text_df = parse_roslynator_text(ROSLYNATOR_RAW_TEXT)
ROSLYNATOR_RESULTS_DF = merge_roslynator_results(xml_df, text_df)
ROSLYNATOR_RESULTS_DF.to_csv(RESULTS_CSV_PATH, index=False)

logger.info(f'Saved raw output: {RAW_OUTPUT_PATH}')
logger.info(f'Saved XML output: {XML_OUTPUT_PATH}')
logger.info(f'Saved CSV results: {RESULTS_CSV_PATH}')
logger.info(f'Total Roslynator findings: {len(ROSLYNATOR_RESULTS_DF)}')

preview_raw_output(ROSLYNATOR_RAW_TEXT, RAW_OUTPUT_PREVIEW_LINES, RAW_OUTPUT_PATH)


## Section 7 — Maintainability Nesting Depth Extraction

Roslynator does not emit nesting depth directly. A custom Roslyn AST analyzer traverses control-flow syntax nodes and computes `max_nesting_depth` per method.


In [ ]:
NESTING_RESULTS_CSV = OUTPUT_PATH / 'nesting_depth_results.csv'

try:
    NESTING_RESULTS_DF = run_nesting_analyzer(
        dotnet_root=DOTNET_ROOT,
        analyzer_dll=ANALYZER_DLL,
        repo=REPO_PATH,
        output_csv=NESTING_RESULTS_CSV,
    )
except Exception as exc:
    logger.error(f'NestingDepthAnalyzer failed: {exc}')
    NESTING_RESULTS_DF = pd.DataFrame(
        columns=['file', 'class', 'method', 'start_line', 'end_line', 'max_nesting_depth', 'status']
    )
    NESTING_RESULTS_DF.to_csv(NESTING_RESULTS_CSV, index=False)

logger.info(f'Saved nesting depth results: {NESTING_RESULTS_CSV}')
logger.info(f'Method rows: {len(NESTING_RESULTS_DF)}')

if not NESTING_RESULTS_DF.empty:
    display(NESTING_RESULTS_DF.head(10))
else:
    print('No nesting depth results produced.')


## Section 8 — Metric Computation

Compute repository-level maintainability nesting depth metrics:

- **Maintainability_Nesting_Depth** = `max(max_nesting_depth(method_i))`
- **Average_Nesting_Depth** = `Σ max_nesting_depth(method_i) / total_methods`


In [ ]:
SUMMARY_DF = compute_summary(NESTING_RESULTS_DF)
SUMMARY_CSV = OUTPUT_PATH / 'maintainability_nesting_depth_summary.csv'
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f'Saved maintainability summary: {SUMMARY_CSV}')
display(SUMMARY_DF)


## Section 9 — Summary Dashboard

Overview of analysis coverage, Roslynator findings, and nesting-depth metrics.


In [ ]:
analyzed_methods = NESTING_RESULTS_DF[NESTING_RESULTS_DF.get('status', pd.Series(dtype=str)) == 'analyzed']
max_depth = SUMMARY_DF.loc[
    SUMMARY_DF['metric_name'] == 'Maintainability_Nesting_Depth', 'metric_value'
].iloc[0]
avg_depth = SUMMARY_DF.loc[
    SUMMARY_DF['metric_name'] == 'Average_Nesting_Depth', 'metric_value'
].iloc[0]
files_failed = count_failed_methods(NESTING_RESULTS_DF)

summary_df = pd.DataFrame(
    [
        {'Metric': 'Total C# Files', 'Value': len(CS_FILES)},
        {'Metric': 'Total Methods', 'Value': len(NESTING_RESULTS_DF)},
        {'Metric': 'Methods Analyzed', 'Value': len(analyzed_methods)},
        {'Metric': 'Maximum Nesting Depth', 'Value': max_depth},
        {'Metric': 'Average Nesting Depth', 'Value': avg_depth},
        {'Metric': 'Files Failed', 'Value': files_failed},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    XML_OUTPUT_PATH,
    RESULTS_CSV_PATH,
    CSHARP_FILES_CSV,
    NESTING_RESULTS_CSV,
    SUMMARY_CSV,
    ERROR_LOG_PATH,
]

print('\nDeliverables:')
for deliverable in deliverables:
    status = 'OK' if deliverable.exists() else 'MISSING'
    print(f'  [{status}] {deliverable}')


## Section 10 — Error Handling

Failures encountered during cloning, validation, Roslynator execution, or AST analysis are appended to `outputs/error_log.txt`.


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 11 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── roslynator_raw_output.txt
├── roslynator_output.xml
├── roslynator_results.csv
├── csharp_files.csv
├── nesting_depth_results.csv
├── maintainability_nesting_depth_summary.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.
